In [1]:
import os

In [2]:
%pwd

'c:\\Users\\p00za\\Desktop\\Collage projects\\Kidney_Disease_Classification_Project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\p00za\\Desktop\\Collage projects\\Kidney_Disease_Classification_Project'

In [5]:
import os
from dotenv import load_dotenv
import mlflow

load_dotenv()

os.environ["MLFLOW_TRACKING_URI"] = os.getenv("MLFLOW_TRACKING_URI")
os.environ["MLFLOW_TRACKING_USERNAME"] = os.getenv("MLFLOW_TRACKING_USERNAME")
os.environ["MLFLOW_TRACKING_PASSWORD"] = os.getenv("MLFLOW_TRACKING_PASSWORD")

In [6]:
import tensorflow as tf

model = tf.keras.models.load_model("artifacts/training/trained_model.h5")


In [7]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model : Path 
    training_data : Path
    all_params : dict
    mlflow_uri : str
    params_image_size : list
    params_batch_size : int

In [8]:
from cnnClassifier.constants import *
from cnnClassifier.utils.Common import read_yaml, create_directories, save_json

In [9]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        param_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(param_filepath)
        create_directories([self.config.artifacts_root])

    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model= "artifacts/training/trained_model.h5",
            training_data = Path("artifacts/data_ingestion/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone"),
            mlflow_uri =  os.getenv("MLFLOW_TRACKING_URI"),
            all_params = self.params,
            params_image_size = self.params.IMAGE_SIZE,
            params_batch_size = self.params.BATCH_SIZE
        )
        return eval_config    

In [10]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

In [11]:
class Evaluation:
    def __init__(
        self,
        config: EvaluationConfig):

        self.config = config

    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale=1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear",
            class_mode="categorical"
            
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        import os

        print("DATASET PATH:", self.config.training_data)
        print("CLASSES FOUND:", os.listdir(self.config.training_data))

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        print(self.valid_generator.class_indices)

    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)


    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = self.model.evaluate(self.valid_generator)
        self.save_score()    
    

    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)



    def log_into_mlflow(self):
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics(
                {"loss": self.score[0], "accuracy": self.score[1]}
            )

            # Model registry does not work with file store
            if tracking_url_type_store !="file":
                # Register the model
                # There are some other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more information:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow
                mlflow.keras.log_model(self.model, "model", registered_model_name="CNNClassifierModel")
            else:
                mlflow.keras.log_model(self.model, "model")



In [12]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(config=eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()
except Exception as e:
    raise e

[2026-05-20 15:08:13,789: INFO: Common: yaml file:config\config.yaml loaded successfully]
[2026-05-20 15:08:13,792: INFO: Common: yaml file:params.yaml loaded successfully]
[2026-05-20 15:08:13,794: INFO: Common: created directory at :artifacts]
DATASET PATH: artifacts\data_ingestion\CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone\CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone
CLASSES FOUND: ['Cyst', 'Normal', 'Stone', 'Tumor']
Found 3732 images belonging to 4 classes.
{'Cyst': 0, 'Normal': 1, 'Stone': 2, 'Tumor': 3}
234/234 [==============================] - 705s 3s/step - loss: 23.5565 - accuracy: 0.2471
[2026-05-20 15:20:01,149: INFO: Common: json file saved at : scores.json]


2026/05/20 15:20:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/20 15:20:07 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


[2026-05-20 15:20:09,289: WARNING: save: Found untraced functions such as _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op while saving (showing 5 of 14). These functions will not be directly callable after loading.]
INFO:tensorflow:Assets written to: C:\Users\p00za\AppData\Local\Temp\tmp2c3y66ll\model\data\model\assets
[2026-05-20 15:20:10,261: INFO: builder_impl: Assets written to: C:\Users\p00za\AppData\Local\Temp\tmp2c3y66ll\model\data\model\assets]


Registered model 'CNNClassifierModel' already exists. Creating a new version of this model...
2026/05/20 15:22:08 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: CNNClassifierModel, version 5
Created version '5' of model 'CNNClassifierModel'.


🏃 View run valuable-panda-196 at: https://dagshub.com/P00za/Kidney_Disease_Classification_Project.mlflow/#/experiments/0/runs/ec361dffd32c437983f758588b8d912a
🧪 View experiment at: https://dagshub.com/P00za/Kidney_Disease_Classification_Project.mlflow/#/experiments/0
